# 01 — Exploration de la potabilité

Objectif : auditer `water_potability.csv` avant modélisation. Aucune imputation globale afin d’éviter la fuite de données.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
DATA_RAW=PROJECT_ROOT/"data"/"raw"; DATA_PROCESSED=PROJECT_ROOT/"data"/"processed"
FIGURES=PROJECT_ROOT/"reports"/"figures"; RESULTS=PROJECT_ROOT/"reports"/"results"; MODELS=PROJECT_ROOT/"models"
for p in [DATA_PROCESSED,FIGURES,RESULTS,MODELS]: p.mkdir(parents=True,exist_ok=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RANDOM_STATE=42
pd.set_option("display.max_columns",100)

In [ ]:
from src.data_cleaning import load_potability,basic_cleaning,data_quality_report,save_processed

## 2. Chargement

In [ ]:
df_raw=load_potability(DATA_RAW/"water_potability.csv")
print("Dimensions :",df_raw.shape)
display(df_raw.head())

## 3. Structure et types

In [ ]:
display(pd.DataFrame({"colonne":df_raw.columns,"dtype":df_raw.dtypes.astype(str).values}))

## 4. Qualité des données

In [ ]:
display(data_quality_report(df_raw))
print("Doublons :",df_raw.duplicated().sum())

## 5. Valeurs manquantes

In [ ]:
m=df_raw.isna().sum().sort_values(ascending=False); display(m[m>0].to_frame("manquants"))
if (m>0).any(): m[m>0].plot.bar(figsize=(8,4),title="Valeurs manquantes"); plt.tight_layout(); plt.show()

## 6. Nettoyage minimal

In [ ]:
df=basic_cleaning(df_raw,target="Potability")
print(df.shape)

## 7. Cible Potability

In [ ]:
counts=df["Potability"].value_counts().sort_index(); pct=(100*df["Potability"].value_counts(normalize=True).sort_index()).round(2)
display(pd.DataFrame({"effectif":counts,"pourcentage":pct}))
counts.plot.bar(figsize=(6,4),title="Répartition Potability"); plt.show()

## 8. Statistiques descriptives

In [ ]:
display(df.describe().T)

## 9. Distributions

In [ ]:
features=[c for c in df.columns if c!="Potability"]
for col in features:
    ax=df[col].dropna().plot.hist(bins=30,figsize=(6,3),title=f"Distribution — {col}"); ax.set_xlabel(col); plt.tight_layout(); plt.show()

## 10. Valeurs atypiques

Inspection seulement : aucune suppression automatique.

In [ ]:
for col in features:
    fig,ax=plt.subplots(figsize=(6,2.5)); ax.boxplot(df[col].dropna(),vert=False); ax.set_title(f"Boxplot — {col}"); plt.tight_layout(); plt.show()

## 11. Corrélations

In [ ]:
corr=df.corr(numeric_only=True); fig,ax=plt.subplots(figsize=(10,8)); im=ax.imshow(corr,cmap="coolwarm",vmin=-1,vmax=1); ax.set_xticks(range(len(corr)),corr.columns,rotation=90); ax.set_yticks(range(len(corr)),corr.index); fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

## 12. Comparaison par classe

In [ ]:
display(df.groupby("Potability")[features].median().T)

## 13. Export nettoyé

In [ ]:
save_processed(df,DATA_PROCESSED/"potability_clean.csv")
print("Export terminé")

## 14. Synthèse à rédiger

Documenter les valeurs manquantes, le déséquilibre de classes, les distributions, les corrélations et les limites du dataset.